# H4: Discussion Quality and Market Reaction

This notebook tests whether the CAR response to small-shareholder victories is larger when the preceding discussion is higher quality. It follows the existing `scripts/reg` workflow: import the regression panel, estimate in Stata with `reghdfe`, and export LaTeX tables with `esttab`.

In [ ]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')

In [ ]:
%%stata

clear all
set more off
set varabbrev off
version 18

* Paths
global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"
cap mkdir "$TABLES"


In [ ]:
%%stata

* Load proposal-level panel built by scripts/reg/reg_small.py
import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear

* Parse date and fixed effects
capture drop day
capture drop year
gen day = date(date, "YMD")
format day %td
gen year = year(day)

* H4 sample and variables
capture drop quality_defined
capture drop small_quality
capture drop nonsmall_win
capture drop sw_x_qdiff
capture drop sw_x_qfull
capture drop nonsw_x_qdiff
capture drop nonsw_x_qfull

gen quality_defined = post_number >= 2 & !missing(concensus_diff) & !missing(concensus_full)
gen small_quality = non_whale_victory_vn == 1 & quality_defined
gen nonsmall_win = 1 - non_whale_victory_vn

gen sw_x_qdiff = non_whale_victory_vn * concensus_diff
gen sw_x_qfull = non_whale_victory_vn * concensus_full
gen nonsw_x_qdiff = nonsmall_win * concensus_diff
gen nonsw_x_qfull = nonsmall_win * concensus_full

* Labels
label var car_created             "\${\it CAR}^{\it Create}_{i}\$"
label var car_end                 "\${\it CAR}^{\it End}_{i}\$"
label var non_whale_victory_vn    "\${\it Small Win}_{i}\$"
label var nonsmall_win            "\${\it NonSmallWin}_{i}\$"
label var concensus_diff          "\${\it \Delta Concensus}_{i}\$"
label var concensus_full          "\${\it Concensus}^{\it full}_{i}\$"
label var sw_x_qdiff              "\${\it Small Win}_{i} \times {\it \Delta Concensus}_{i}\$"
label var sw_x_qfull              "\${\it Small Win}_{i} \times {\it Concensus}^{\it full}_{i}\$"
label var nonsw_x_qdiff           "\${\it NonSmallWin}_{i} \times {\it \Delta Concensus}_{i}\$"
label var nonsw_x_qfull           "\${\it NonSmallWin}_{i} \times {\it Concensus}^{\it full}_{i}\$"


In [ ]:
%%stata

******************************************************
* FEASIBILITY CHECK - RUN FIRST
******************************************************

count if quality_defined
local n_quality = r(N)
count if quality_defined & !missing(car_created)
local n_car_created = r(N)
count if quality_defined & !missing(car_end)
local n_car_end = r(N)
count if small_quality
local n_small_quality = r(N)
count if small_quality & !missing(car_created)
local n_small_car_created = r(N)
count if small_quality & !missing(car_end)
local n_small_car_end = r(N)

display ""
display "H4 feasibility counts"
display "Proposals with >=2 discussion posts and quality defined: `n_quality'"
display "  of which non-missing CARCreate: `n_car_created'"
display "  of which non-missing CAREnd: `n_car_end'"
display "SmallWin = 1 and quality defined: `n_small_quality'"
display "SmallWin = 1, quality defined, and non-missing CARCreate: `n_small_car_created'"
display "SmallWin = 1, quality defined, and non-missing CAREnd: `n_small_car_end'"

capture file close h4feas
file open h4feas using "$TABLES/h4_feasibility.tex", write replace
file write h4feas "\begin{tabular}{lc}" _n
file write h4feas "\toprule" _n
file write h4feas "Cell & Count \\\\" _n
file write h4feas "\midrule" _n
file write h4feas "Proposals with $\geq 2$ discussion posts (quality defined) & `n_quality' \\\\" _n
file write h4feas "\quad of which non-missing $\mathrm{CARCreate}$ & `n_car_created' \\\\" _n
file write h4feas "\quad of which non-missing $\mathrm{CAREnd}$ & `n_car_end' \\\\" _n
file write h4feas "$\mathrm{SmallWin}_i=1$ and quality defined & `n_small_quality' \\\\" _n
file write h4feas "$\mathrm{SmallWin}_i=1$, quality defined, and non-missing $\mathrm{CARCreate}$ & `n_small_car_created' \\\\" _n
file write h4feas "$\mathrm{SmallWin}_i=1$, quality defined, and non-missing $\mathrm{CAREnd}$ & `n_small_car_end' \\\\" _n
file write h4feas "\bottomrule" _n
file write h4feas "\end{tabular}" _n
file close h4feas


In [ ]:
%%stata

******************************************************
* MAIN H4 INTERACTION REGRESSIONS
******************************************************

eststo clear

reghdfe car_created non_whale_victory_vn sw_x_qdiff concensus_diff if quality_defined, absorb(year categories)
eststo qdiff_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end non_whale_victory_vn sw_x_qdiff concensus_diff if quality_defined, absorb(year categories)
eststo qdiff_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_created non_whale_victory_vn sw_x_qfull concensus_full if quality_defined, absorb(year categories)
eststo qfull_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end non_whale_victory_vn sw_x_qfull concensus_full if quality_defined, absorb(year categories)
eststo qfull_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

esttab                                                           ///
    qdiff_created qdiff_end qfull_created qfull_end              ///
    using "$TABLES/reg_h4_interaction.tex", replace              ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    keep(non_whale_victory_vn sw_x_qdiff sw_x_qfull concensus_diff concensus_full) ///
    order(non_whale_victory_vn sw_x_qdiff sw_x_qfull concensus_diff concensus_full) ///
    mgroups("Quality $=\Delta$ Consensus" "Quality $=$ Consensus level", ///
            span pattern(1 0 1 0) prefix(\multicolumn{@span}{c}{) suffix(}) ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} \\ \cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_proposal fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
        labels("Category FE" "Year FE" "Observations" "Adjusted R²"))


In [ ]:
%%stata

******************************************************
* BACKUP MEDIAN-SPLIT REGRESSIONS
******************************************************

capture drop high_qfull
summ concensus_full if quality_defined, detail
gen high_qfull = concensus_full > r(p50) if quality_defined & !missing(concensus_full)

eststo clear

reghdfe car_created non_whale_victory_vn if quality_defined & high_qfull == 0, absorb(year categories)
eststo created_low
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_created non_whale_victory_vn if quality_defined & high_qfull == 1, absorb(year categories)
eststo created_high
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end non_whale_victory_vn if quality_defined & high_qfull == 0, absorb(year categories)
eststo end_low
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end non_whale_victory_vn if quality_defined & high_qfull == 1, absorb(year categories)
eststo end_high
estadd local fe_proposal "Y"
estadd local fe_time "Y"

esttab                                                           ///
    created_low created_high end_low end_high                    ///
    using "$TABLES/reg_h4_split.tex", replace                    ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    keep(non_whale_victory_vn)                                   ///
    mgroups("\${\it CAR}^{\it Create}_{i}\$" "\${\it CAR}^{\it End}_{i}\$", ///
            span pattern(1 0 1 0) prefix(\multicolumn{@span}{c}{) suffix(}) ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{Low quality} & \multicolumn{1}{c}{High quality} & \multicolumn{1}{c}{Low quality} & \multicolumn{1}{c}{High quality} \\ \cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_proposal fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
        labels("Category FE" "Year FE" "Observations" "Adjusted R²"))


In [ ]:
%%stata

******************************************************
* PLACEBO: NON-SMALL-WIN INTERACTION
******************************************************

eststo clear

reghdfe car_created nonsmall_win nonsw_x_qdiff concensus_diff if quality_defined, absorb(year categories)
eststo qdiff_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end nonsmall_win nonsw_x_qdiff concensus_diff if quality_defined, absorb(year categories)
eststo qdiff_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_created nonsmall_win nonsw_x_qfull concensus_full if quality_defined, absorb(year categories)
eststo qfull_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end nonsmall_win nonsw_x_qfull concensus_full if quality_defined, absorb(year categories)
eststo qfull_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

esttab                                                           ///
    qdiff_created qdiff_end qfull_created qfull_end              ///
    using "$TABLES/reg_h4_interaction_placebo.tex", replace      ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    keep(nonsmall_win nonsw_x_qdiff nonsw_x_qfull concensus_diff concensus_full) ///
    order(nonsmall_win nonsw_x_qdiff nonsw_x_qfull concensus_diff concensus_full) ///
    mgroups("Quality $=\Delta$ Consensus" "Quality $=$ Consensus level", ///
            span pattern(1 0 1 0) prefix(\multicolumn{@span}{c}{) suffix(}) ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} \\ \cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_proposal fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
        labels("Category FE" "Year FE" "Observations" "Adjusted R²"))


In [ ]:
%%stata

******************************************************
* WINSORIZED ROBUSTNESS CHECK
******************************************************

preserve
foreach v in car_created car_end concensus_diff concensus_full {
    quietly summ `v', detail
    replace `v' = r(p99) if `v' > r(p99) & !missing(`v')
}

capture drop sw_x_qdiff_w sw_x_qfull_w
gen sw_x_qdiff_w = non_whale_victory_vn * concensus_diff
gen sw_x_qfull_w = non_whale_victory_vn * concensus_full
label var sw_x_qdiff_w "\${\it Small Win}_{i} \times {\it \Delta Concensus}_{i}\$"
label var sw_x_qfull_w "\${\it Small Win}_{i} \times {\it Concensus}^{\it full}_{i}\$"

eststo clear

reghdfe car_created non_whale_victory_vn sw_x_qdiff_w concensus_diff if quality_defined, absorb(year categories)
eststo qdiff_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end non_whale_victory_vn sw_x_qdiff_w concensus_diff if quality_defined, absorb(year categories)
eststo qdiff_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_created non_whale_victory_vn sw_x_qfull_w concensus_full if quality_defined, absorb(year categories)
eststo qfull_created
estadd local fe_proposal "Y"
estadd local fe_time "Y"

reghdfe car_end non_whale_victory_vn sw_x_qfull_w concensus_full if quality_defined, absorb(year categories)
eststo qfull_end
estadd local fe_proposal "Y"
estadd local fe_time "Y"

esttab                                                           ///
    qdiff_created qdiff_end qfull_created qfull_end              ///
    using "$TABLES/reg_h4_interaction_winsorized.tex", replace   ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    keep(non_whale_victory_vn sw_x_qdiff_w sw_x_qfull_w concensus_diff concensus_full) ///
    order(non_whale_victory_vn sw_x_qdiff_w sw_x_qfull_w concensus_diff concensus_full) ///
    mgroups("Quality $=\Delta$ Consensus" "Quality $=$ Consensus level", ///
            span pattern(1 0 1 0) prefix(\multicolumn{@span}{c}{) suffix(}) ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it Create}_{i}\$} & \multicolumn{1}{c}{\${\it CAR}^{\it End}_{i}\$} \\ \cmidrule(lr){2-3}\cmidrule(lr){4-5} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} & \multicolumn{1}{c}{(4)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_proposal fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
        labels("Category FE" "Year FE" "Observations" "Adjusted R²"))
restore
